# Technologies.csv Generation — Reality Access Scenario — Norte Amazónica 2025

Generates one `Technologies.csv` per cluster (C1–C5) for EnergyScope ESMC, **reality access scenario**.
Base: reality output (`../reality/output_energyscope/C{k}/Technologies.csv`).
Only values documented below are overwritten; all other rows are inherited unchanged from reality.

## Parameters — reality-run outputs

`GENSET_DIESEL`, `DEC_DIRECT_ELEC` and `DEC_BOILER_GAS` annual production floors per cluster. These
values are the `F_year` dispatch outputs of the solved reality-scenario EnergyScope run — they cannot
be recomputed from this pipeline's input data, so they're supplied here as parameters rather than
derived. They freeze the supply already serving connected households at the reality-run level, so the
access-scenario optimizer can't erode it while sizing new capacity for the newly-connected ones.

In [1]:
GENSET_DIESEL_FMIN_PROD   = {1: 0.0,      2: 0.0,      3: 103.859134, 4: 25.652892, 5: 55.408515}
DEC_DIRECT_ELEC_FMIN_PROD = {1: 0.365641, 2: 0.0,      3: 1.613232,   4: 0.0,       5: 0.689686}
DEC_BOILER_GAS_FMIN_PROD  = {1: 0.0,      2: 0.017168, 3: 0.882362,   4: 0.650953,  5: 0.882362}

In [2]:
import os
import pandas as pd

OUT_DIR     = "output_energyscope"
REALITY_DIR = "../reality/output_energyscope"

CLUSTERS = {
    1: ["Exaltación", "Reyes", "Santa_Rosa_Beni", "Ixiamas"],
    2: ["Bolpebra"],
    3: ["Guayaramerín", "Riberalta", "Puerto_Gonzalo_Moreno"],
    4: ["Bella_Flor", "Filadelfia", "Ingavi", "Nueva_Esperanza", "Porvenir",
        "Puerto_Rico", "San_Lorenzo", "San_Pedro", "Santa_Rosa_Pando",
        "Santos_Mercado", "Sena", "Villa_Nueva"],
    5: ["Cobija"],
}

# Load reality output as base — only PV_HS/HS_DIESEL/BATT_HS/GENSET_DIESEL rows are overwritten
reality = {}
for k in range(1, 6):
    path = os.path.join(REALITY_DIR, f"C{k}", "Technologies.csv")
    df = pd.read_csv(path, sep=";")
    df["Technologies param"] = df["Technologies param"].str.strip()
    reality[k] = df
print("Loaded reality base for C1–C5")

Loaded reality base for C1–C5


## 1. Dispersed-household demand

Households that remain dispersed keep meeting their electricity needs off-grid. Their annual demand
becomes the `HS_DIESEL` and `PV_HS` production ceiling (`f_max_prod`, dual cap) below, and for C2
also the `PV_HS` production floor (`f_min_prod`).

**Source: `output/2024/cluster_summary.csv`, column `demande_dispersee_GWh`.** This is the
greedy-tree-with-definitive-rejection classification (71 dispersed communities out of 697). It
replaces the earlier method — summing `HH` over the `dispersed` flag of
`community_breakeven_detail_BC.csv` and multiplying by 1168.9 kWh/HH — which implements the
superseded independent-breakeven classification (253 dispersed communities) and yields volumes
about 8.7x larger. The published catalogue in `Data/2025/reality_access` has been on the
greedy-tree figures since 2026-08-17.

In [3]:
GIS_OUT = "../../analyse_GIS_phase2_projections/output"

cluster_summary = pd.read_csv(f"{GIS_OUT}/2024/cluster_summary.csv").set_index("Cluster")

dispersed_demand_gwh = {k: float(cluster_summary.loc[f"C{k}", "demande_dispersee_GWh"])
                        for k in range(1, 6)}

for k in range(1, 6):
    print(f"C{k}: dispersed_demand = {dispersed_demand_gwh[k]:.4f} GWh")

C1: dispersed_demand = 0.1794 GWh
C2: dispersed_demand = 0.0000 GWh
C3: dispersed_demand = 0.1107 GWh
C4: dispersed_demand = 0.2419 GWh
C5: dispersed_demand = 0.0000 GWh


## 2. Off-grid capacities — PV_HS, HS_DIESEL, BATT_HS

`f_min` for the three off-grid technologies comes directly from `share_dispersion_final_BC.csv`
(columns already in GW / GWh, no reconversion); `f_max` is left uncapped (`1e15`) — these are
existing-plus-buildable capacities in the access scenario, not locked like in `reality`.

Read from the live GIS pipeline (`analyse_GIS_phase2_projections/output/`), not from the frozen
v1 folder: the two versions of this file differ (household counts 20 / 0 / 10 / 41 / 0 against
79 / 5 / 29 / 129 / 0), and the published catalogue is on the live one.

In [4]:
share_disp = pd.read_csv(f"{GIS_OUT}/share_dispersion_final_BC.csv")
share_disp = share_disp.set_index("Cluster")

pv_hs_fmin     = {k: share_disp.loc[f"C{k}", "f_min_PV_HS_GW"]     for k in range(1, 6)}
hs_diesel_fmin = {k: share_disp.loc[f"C{k}", "f_min_HS_DIESEL_GW"] for k in range(1, 6)}
batt_hs_fmin   = {k: share_disp.loc[f"C{k}", "f_min_BATT_HS_GWh"]  for k in range(1, 6)}

print(f"{'':8} {'PV_HS f_min':>14} {'HS_DIESEL f_min':>18} {'BATT_HS f_min':>16}")
for k in range(1, 6):
    print(f"C{k}       {pv_hs_fmin[k]:>14.6g} {hs_diesel_fmin[k]:>18.6g} {batt_hs_fmin[k]:>16.6g}")

            PV_HS f_min    HS_DIESEL f_min    BATT_HS f_min
C1              3.6e-06          1.335e-05         8.85e-06
C2                    0                  0                0
C3              8.4e-07           5.78e-06         2.07e-06
C4              4.9e-07          2.407e-05          1.2e-06
C5                    0                  0                0


## 3. Assemble and save

In [5]:
for k in range(1, 6):
    df = reality[k].copy()

    for tech, fmin in [("PV_HS", pv_hs_fmin[k]), ("HS_DIESEL", hs_diesel_fmin[k]), ("BATT_HS", batt_hs_fmin[k])]:
        mask = df["Technologies param"] == tech
        df.loc[mask, "f_min"] = fmin
        df.loc[mask, "f_max"] = 1e15

    # Dual f_max_prod cap: PV_HS and HS_DIESEL both capped at the dispersed-household demand, so
    # the pair cannot serve more than the households that stay off-grid.
    df.loc[df["Technologies param"] == "PV_HS", "f_min_prod"] = 0.0
    df.loc[df["Technologies param"] == "PV_HS", "f_max_prod"] = dispersed_demand_gwh[k]
    if k == 2:
        df.loc[df["Technologies param"] == "PV_HS", "f_min_prod"] = dispersed_demand_gwh[2]

    df.loc[df["Technologies param"] == "HS_DIESEL", "f_min_prod"] = 0.0
    df.loc[df["Technologies param"] == "HS_DIESEL", "f_max_prod"] = dispersed_demand_gwh[k]

    df.loc[df["Technologies param"] == "BATT_HS", "f_min_prod"] = 0.0
    df.loc[df["Technologies param"] == "BATT_HS", "f_max_prod"] = 1e15

    df.loc[df["Technologies param"] == "GENSET_DIESEL",   "f_min_prod"] = GENSET_DIESEL_FMIN_PROD[k]
    df.loc[df["Technologies param"] == "DEC_DIRECT_ELEC",  "f_min_prod"] = DEC_DIRECT_ELEC_FMIN_PROD[k]
    df.loc[df["Technologies param"] == "DEC_BOILER_GAS",   "f_min_prod"] = DEC_BOILER_GAS_FMIN_PROD[k]

    out_path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    df.to_csv(out_path, sep=";", index=False)

print("Saved Technologies.csv for C1–C5")

Saved Technologies.csv for C1–C5

## 4. Verification

In [6]:
clusters_out = {}
for k in range(1, 6):
    path = os.path.join(OUT_DIR, f"C{k}", "Technologies.csv")
    df = pd.read_csv(path, sep=";")
    df["Technologies param"] = df["Technologies param"].str.strip()
    clusters_out[k] = df

def lookup(df, tech, col):
    row = df.loc[df["Technologies param"] == tech, col]
    return float(row.values[0]) if len(row) else float("nan")

header = f"{'Technology':<32}" + "".join(f"  C{k:>11}" for k in range(1, 6))
sep    = "-" * len(header)

print("=== PV_HS / HS_DIESEL / BATT_HS — f_min (share_dispersion_final_BC.csv) ===")
print(header); print(sep)
for tech in ["PV_HS", "HS_DIESEL", "BATT_HS"]:
    vals = [lookup(clusters_out[k], tech, "f_min") for k in range(1, 6)]
    print(f"{tech+' f_min':<32}" + "".join(f"  {v:>11.6f}" for v in vals))

print()
print("=== Confirm the override comes from share_dispersion_final_BC.csv, not from reality ===")
print(header); print(sep)
for tech in ["PV_HS", "HS_DIESEL", "BATT_HS"]:
    reality_vals = [lookup(reality[k], tech, "f_min") for k in range(1, 6)]
    access_vals  = [lookup(clusters_out[k], tech, "f_min") for k in range(1, 6)]
    print(f"{tech+' f_min (reality, inherited base)':<40}" + "".join(f"  {v:>11.6g}" for v in reality_vals))
    print(f"{tech+' f_min (access, written)':<40}" + "".join(f"  {v:>11.6g}" for v in access_vals))
    not_inherited = ["OK" if abs(access_vals[i] - reality_vals[i]) > 1e-9 or reality_vals[i] == 0
                      else "SAME AS REALITY" for i in range(5)]
    print(f"{'  overridden (not inherited)?':<40}" + "".join(f"  {s:>11}" for s in not_inherited))
    print()

print("=== HS_DIESEL / PV_HS — production floor/ceiling (dispersed-household demand) ===")
print(header); print(sep)
vals = [lookup(clusters_out[k], "HS_DIESEL", "f_max_prod") for k in range(1, 6)]
print(f"{'HS_DIESEL f_max_prod':<32}" + "".join(f"  {v:>11.4f}" for v in vals))
vals = [lookup(clusters_out[k], "PV_HS", "f_min_prod") for k in range(1, 6)]
print(f"{'PV_HS f_min_prod':<32}" + "".join(f"  {v:>11.4f}" for v in vals))

print()
print("=== GENSET_DIESEL / DEC_DIRECT_ELEC / DEC_BOILER_GAS f_min_prod (run-output parameters) ===")
print(header); print(sep)
for tech in ["GENSET_DIESEL", "DEC_DIRECT_ELEC", "DEC_BOILER_GAS"]:
    vals = [lookup(clusters_out[k], tech, "f_min_prod") for k in range(1, 6)]
    print(f"{tech+' f_min_prod':<32}" + "".join(f"  {v:>11.6f}" for v in vals))

print()
print("=== Cross-check: no tech has f_min_prod > 0 with f_max == 0 ===")
for k in range(1, 6):
    bad = clusters_out[k][(clusters_out[k]["f_min_prod"] > 0) & (clusters_out[k]["f_max"] <= 0)]
    status = "OK" if bad.empty else f"FAIL: {bad['Technologies param'].tolist()}"
    print(f"  C{k}: {status}")

=== PV_HS / HS_DIESEL / BATT_HS — f_min (share_dispersion_final_BC.csv) ===


Technology                        C          1  C          2  C          3  C          4  C          5
------------------------------------------------------------------------------------------------------
PV_HS f_min                          0.000004     0.000000     0.000001     0.000000     0.000000
HS_DIESEL f_min                      0.000013     0.000000     0.000006     0.000024     0.000000
BATT_HS f_min                        0.000009     0.000000     0.000002     0.000001     0.000000

=== Confirm the override comes from share_dispersion_final_BC.csv, not from reality ===
Technology                        C          1  C          2  C          3  C          4  C          5
------------------------------------------------------------------------------------------------------
PV_HS f_min (reality, inherited base)      8.6425e-05   1.1225e-05   6.9275e-05    7.095e-05    8.775e-06
PV_HS f_min (access, written)                 3.6e-06            0      8.4e-07      4.9e-07      